# NSF Downloads Pipeline

The NSF (as of April 2026) provides zipped directories with one JSON file per grant, per year, updated regularly. The data can be found at https://www.nsf.gov/awardsearch/download-awards. The robots.txt file for NSF.gov disallows the /awardsearch/* path so we manually download the years of interest to a relavent directory.

This notebook will:

1. Read the designated directory (unless configured not to download). This directory should be separate from, or nested below the final data outputs directory for convenience and clarity.
2. Unzip each folder into the same directory
3. Process every discovered JSON file in the directory and its children into the OIC data model
4. Write out a json-lines format file in the OIC datamodel to the designated data output directory. Output location MUST be a directory to allow for batching of the outputs into multiple files

In [1]:
from pathlib import Path
from zipfile import ZipFile
import json
from itertools import batched
from attrs import asdict
import polars as pl
from typing import List, Dict, Any
from datetime import datetime, date
from oic_scrape.items import AwardParticipant, AwardItem


In [2]:
DATA_INPUT_DIRECTORY = Path("../data/202604/nsf")
FUNDER_ORG_NAME = "National Science Foundation"
FUNDER_ROR_ID = "https://ror.org/021nxhr62"
OUTPUT_LOCATION = Path("../data/202604")
OUTPUT_FORMAT = "jsonl"

In [3]:
awards = []
zip_files = DATA_INPUT_DIRECTORY.glob("*.zip")
for file in zip_files:
    with ZipFile(file) as z:
        jfiles = z.namelist()
        for jfile in jfiles:
            j = json.load(z.open(jfile))
            award = AwardItem(
                _crawled_at=datetime.now(),
                source='https://www.nsf.gov/awardsearch/download-awards',
                grant_id=f"nsf::{j.get('awd_id')}",
                funder_org_name=FUNDER_ORG_NAME,
                funder_org_ror_id=FUNDER_ROR_ID,
                recipient_org_name=j['inst']['inst_name'],
                recipient_org_location=f"{j['inst'].get('inst_city_name')}, {j['inst'].get('inst_state_name')}, {j['inst'].get('inst_country_name')}",
                named_participants=[
                    AwardParticipant(
                        full_name=pi['pi_full_name'],
                        first_name=pi.get('pi_first_name'),
                        last_name=pi.get('pi_last_name'),
                        grant_role=pi.get('pi_role'),
                        identifiers=dict(
                            nsf_id=pi.get('nsf_id'),
                            email=pi.get('pi_email_addr')
                        ),
                        is_pi=True if pi.get('pi_role') == 'Principal Investigator' else False,
                    )
                    for pi in j['pi']
                ] if j.get('pi') else None,
                grant_year=date.fromisoformat(j['awd_eff_date']).year,
                grant_start_date=date.fromisoformat(j['awd_eff_date']),
                grant_end_date=date.fromisoformat(j['awd_exp_date']),
                award_amount=j.get('awd_amount'),
                award_currency='USD',
                grant_title=j['awd_titl_txt'],
                grant_description=j.get('awd_abstract_narration'),
                # program_of_funder=j['pgm_ele']['pgm_ele_name'],
                raw_source_data = json.dumps(j),
            )
            awards.append(asdict(award))


In [4]:
for i, chunk in enumerate(batched(awards, 100000)):
    pl.DataFrame(chunk).write_ndjson(OUTPUT_LOCATION / f'nsf_{i:04}.jsonl')

In [5]:
[print(file.name) for file in zip_files]

[]